In [24]:
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage

In [13]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    openai_api_key="e97ee307-a791-4e06-ade1-df4b9d032eed",
    openai_api_base="https://aihub-api.sktelecom.com/aihub/v2/sandbox",
    temperature=0,
)

In [17]:
neo4j_schema = """
Node properties:
- **요금제**
  - `장애인혜택제공여부`: STRING Available options: ['N', 'Y']
  - `고객유형별가입가능여부`: STRING Available options: ['null', 'true', 'false']
  - `장애인부가통화추가제공량`: INTEGER Min: 0, Max: 250
  - `문자대상`: STRING Available options: ['[]']
  - `가입가능최소나이계산기준`: STRING Available options: ['null', '일기준', '월기준']
  - `다이렉트플랜가입가능여부`: STRING Available options: ['Y', 'null']
  - `가입가능최대나이`: INTEGER Min: 12, Max: 999
  - `개인고객세부유형별가입가능여부`: STRING Available options: ['null', 'true']
  - `최대나이계산기준`: STRING Available options: ['null', '월기준']
  - `T지원금약정동시가입가능여부`: STRING Available options: ['Y', 'null']
  - `선택약정동시가입가능여부`: STRING Available options: ['Y', 'null']
  - `군인전용요금제여부`: STRING Available options: ['null', 'Y']
  - `가입가능최소나이`: INTEGER Min: 0, Max: 80
  - `동일명의가입가능여부`: STRING Available options: ['null', 'false']
  - `최소충전금액`: FLOAT Min: 0.0, Max: 1000.0
  - `최대충전금액`: FLOAT Min: 0.0, Max: 20000.0
  - `시니어대상데이터소진후최대금액및속도제한적용`: STRING Available options: ['N', 'Y']
  - `기본제공데이터용량`: FLOAT Min: 0.0, Max: 99999.0
  - `충전서비스대상여부`: STRING Available options: ['N', 'Y']
  - `기본제공데이터중공유가능용량`: FLOAT Min: 0.0, Max: 120.0
  - `데이터리필쿠폰선물가능여부`: STRING Available options: ['Y', 'null']
  - `데이터선물받기가능여부`: STRING Available options: ['Y', 'null']
  - `기본제공데이터중mvoip용량`: FLOAT Min: 0.0, Max: 99999.0
  - `데이터리필가능용량`: FLOAT Min: 0.29296875, Max: 99999.0
  - `데이터소진후데이터제공속도`: FLOAT Min: 0.0, Max: 5.0
  - `최대데이터선물가능용량`: FLOAT Min: 0.0, Max: 2.0
  - `리필비율한도`: FLOAT Min: 0.0, Max: 0.2
  - `문자제공량`: INTEGER Min: 50, Max: 99999
  - `데이터소진후최대금액및속도제한적용`: STRING Available options: ['N', 'Y']
  - `음성통화제공량`: INTEGER Min: 0, Max: 99999
  - `영상및부가통화제공량`: INTEGER Min: 0, Max: 400
  - `지정번호통화제공량`: STRING Available options: ['null', '2.0']
  - `부가세제외월정액`: INTEGER Min: 8000, Max: 113637
  - `net가격`: INTEGER Min: 8000, Max: 113637
  - `청구방법`: STRING Available options: ['후불']
  - `운영상태`: STRING Available options: ['운영', '가입중단']
  - `선택약정할인포함부가세제외월정액`: INTEGER Min: 7425, Max: 125000
  - `월정액`: INTEGER Min: 8800, Max: 125000
  - `상품코드매핑`: LIST Min Size: 1, Max Size: 1
  - `영문상품명`: STRING Example: "Direct5G 38"
  - `상품분류`: STRING Available options: ['상품 > 기본요금제 > 휴대폰 요금제', '상품 > 기본요금제']
  - `상품명`: STRING Example: "다이렉트5G 38"
  - `마케팅키워드`: LIST Min Size: 1, Max Size: 39
  - `고유ID`: STRING Example: "PA00000001"
  - `상품설명`: STRING Example: "월 15GB 데이터를 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입 가능한"
  - `라인업`: STRING Example: "다이렉트플랜"
  - `상품가입조건`: STRING Example: "T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능"
- **요금제그룹**
  - `그룹명`: STRING Available options: ['TING_PRCPLN', 'ONESVC_SILVER_PROD', 'NETFLIX_PRCPLN', '디즈니+ 요금제', '스마트기기 요금제', '유튜프 프리미엄 요금제']
- **부가서비스**
  - `부가서비스명`: STRING Example: "0플랜 심야 데이터 무제한"
  - `부가서비스ID`: STRING Example: "NA00006162"
Relationship properties:

The relationships:
(:요금제)-[:해지이전해지]->(:부가서비스)
(:요금제)-[:가입동시해지]->(:부가서비스)
(:요금제)-[:가입이전해지]->(:부가서비스)
(:요금제)-[:해지동시해지]->(:부가서비스)
(:요금제)-[:할인]->(:부가서비스)
(:요금제)-[:속함]->(:요금제그룹)
(:요금제)-[:데이터충전]->(:부가서비스)
"""

In [18]:
prompt_by_chatgpt = f"""당신은 뛰어난 대화형 AI 어시스턴트입니다.  
아래에 SK텔레콤 모바일 요금제 정보를 담고 있는 Neo4j 그래프 DB의 스키마가 주어집니다.  
스키마를 참고하여, 실제 SK텔레콤 고객이 고객센터 챗봇이나 검색창에 입력할 법한 자연어 질문을 **50개** 생성하세요.  

---  
## Neo4j 스키마  
{neo4j_schema}
---  

### 요구사항  
1. **카테고리 다양성**: 가격·데이터·통화·문자·부가서비스·단말기호환·할인·지역·약정·해지·요금제 전환 등 여러 측면을 골고루 포함  
2. **질문 형태**:  
   - 일반 고객이 쓰는 자연어(예: “~얼마인가요?”, “~가능한가요?”, “어떻게 신청하나요?” 등)  
   - 최대한 구체적이고 명확하게  
3. **출력 형식**:  
   1번부터 50번까지 순서 있는 번호 리스트  
   각 항목은 “?”로 끝나는 단일 문장  
4. **스키마 외 추가 가정 불필요**: 스키마에 없는 속성은 사용하지 말 것  

이제 50개의 질문을 생성하세요."""

In [20]:
prompt_by_claude = f"""## 역할
당신은 SK텔레콤의 모바일 요금제 데이터를 저장한 Neo4j 그래프 데이터베이스에 대해 사용자가 실제로 질문할 만한 현실적이고 다양한 질의들을 생성하는 전문가입니다.

## 그래프 DB 스키마
{neo4j_schema}

## 지시사항
위의 스키마를 바탕으로 SK텔레콤 고객, 직원, 분석가들이 실제로 물어볼 만한 질문 50개를 생성하세요.

### 질문 생성 기준:
1. **다양한 사용자 관점 고려**:
   - 일반 고객 (요금제 선택, 비교)
   - 기업 고객 (단체 요금제, 비용 최적화)
   - SK텔레콤 직원 (운영, 분석)
   - 경영진/분석가 (전략, 수익성)

2. **질의 복잡도 다양화**:
   - 단순 조회 (20%)
   - 비교/필터링 (30%)
   - 집계/통계 (25%)
   - 복합 분석 (25%)

3. **비즈니스 시나리오 반영**:
   - 요금제 추천
   - 경쟁사 비교
   - 수익성 분석
   - 고객 세그먼트 분석
   - 마케팅 전략

4. **한국어 자연어로 작성**:
   - 구어체와 문어체 혼용
   - 실제 고객이 쓸 법한 표현 사용

### 출력 형식:
각 질문에 대해 다음 형식으로 작성:
질문 [번호]: [질문 내용]

이제 50개의 질문을 생성해주세요."""

In [21]:
prompt_by_gemini = f"""# 역할
당신은 SK텔레콤의 모바일 요금제 데이터를 분석하고 사용자 경험을 개선하는 임무를 맡은 AI 전문가입니다. 당신의 주요 임무는 주어진 데이터베이스 스키마를 기반으로 사용자들이 궁금해할 만한 질문들을 예측하고 생성하는 것입니다.
# 목표
주어진 Neo4j 그래프 데이터베이스 스키마를 기반으로, 사용자들이 SK텔레콤 모바일 요금제에 대해 궁금해할 만한 현실적이고 다양한 질문 50개를 생성합니다.
# 배경
생성된 질문 목록은 향후 개발될 AI 챗봇의 학습 데이터, 웹사이트의 '자주 묻는 질문(FAQ)' 섹션 구성, 또는 사용자 검색 의도 분석 자료로 활용될 예정입니다. 따라서 질문은 실제 사용자의 관점에서 자연스럽고 구체적이어야 합니다.
# 주어진 Neo4j 그래프 DB 스키마
{neo4j_schema}
# 지시사항
1. 위 스키마를 기반으로 총 50개의 사용자 질문을 생성하세요.
2. 모든 질문은 주어진 스키마의 노드, 속성, 관계를 조합하여 답변이 가능한 형태여야 합니다.
3. 실제 사용자가 검색창이나 챗봇에 입력할 법한 자연스러운 구어체 문장으로 작성하세요. (예: "XX 요금제 가격이 얼마인가요?" O, "Plan 노드의 price 속성 조회" X)
4. 다양한 사용자 의도를 반영하기 위해 아래 7가지 질문 유형을 모두 포함하여 균형 있게 생성해주세요.
  A. 단순 정보 조회형: 특정 요금제의 가격, 데이터양 등 단일 정보를 묻는 질문
  B. 비교/대조형: 두 개 이상의 요금제를 비교하거나 차이점을 묻는 질문
  C. 추천/조건 기반형: 사용자의 조건(예: 데이터 사용량, 월 예산)에 맞는 요금제를 추천해달라는 질문
  D. 혜택/기능 중심형: 특정 혜택(예: wavve, 데이터 공유)이 포함된 요금제를 찾는 질문
  E. 할인/결합 관련형: 특정 할인(예: 가족 결합)이 가능한 요금제를 묻는 질문
  F. 특정 사용자 그룹 타겟형: 특정 연령대나 그룹(예: 청소년, 어르신)을 위한 요금제를 찾는 질문
  G. 복합/다중 조건형: 두 개 이상의 조건을 동시에 만족하는 요금제를 찾는 질문
  H. 스키마에 명시되지 않은 정보(예: 특정 스마트폰 기종과의 결합, 멤버십 포인트 사용, 로밍)에 대한 질문은 생성하지 마세요.
# 출력 형식
  - 질문 목록을 번호(1~50)를 붙여서 리스트 형태로 제공해주세요.
  - 각 질문 뒤에 괄호를 사용하여 어떤 질문 유형(A~G)에 해당하는지 표기해주세요.
  - 예시: 1. 5G 베이직 요금제는 한 달에 얼마야? (A)"""

In [23]:
prompt_by_perplexity = f"""## 프롬프트 지시사항
다음 Neo4j 그래프 스키마를 기반으로 SK텔레콤 모바일 요금제에 대한 사용자 질문 50개를 생성하세요. 
질문은 다양한 유형(계층적 탐색, 관계 분석, 속성 비교 등)을 포함해야 합니다.

### 그래프 스키마 구조
{neo4j_schema}

### 질문 생성 가이드라인
1. **계층적 탐색** (15문항):  
   - 요금제 → 지원기기 → 프로모션 경로 질문  
   *예: "5GX 프리미엄 요금제를 사용하는 기기 중 현재 프로모션이 적용된 기기는?"*

2. **속성 비교** (10문항):  
   - 월비용/데이터 용량/네트워크 타입 기준 필터링  
   *예: "월 40,000원 이하 요금제 중 5G 지원되는 요금제는?"*

3. **관계 분석** (15문항):  
   - 특정 관계(SUBSCRIBES/SUPPORTS/INCLUDES) 활용  
   *예: "갤럭시 S24와 호환되는 요금제의 평균 월비용은?"*

4. **패턴 분석** (10문항):  
   - 그래프 구조 기반 트렌드 분석  
   *예: "가장 많은 프로모션을 포함하는 요금제 유형은?"*

### 출력 형식
{{ "questions": [{{"type": "계층적 탐색", "question": "생성된 질문1"}}, {{"type": "속성 비교", "question": "생성된 질문2"}}, ...] }}

### 주의사항
- 생성 시 실제 그래프 노드/관계명(Plan, SUBSCRIBES 등) 반드시 사용
- 속성값은 한국어로 작성(ex: "5GX 플래티넘")
- 통계적 분석 질문 최소 7문항 포함
- 질문 간 중복성 배제"""

In [26]:
generated_questions = []
for system_prompt in [prompt_by_chatgpt, prompt_by_claude, prompt_by_gemini, prompt_by_perplexity]:
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content="50개의 예상 질문을 작성해줘"),
    ]
    generated_questions.append(model(messages))

In [27]:
for x in generated_questions:
    print(x)
    print("\n\n")

content='1. SK텔레콤의 기본 요금제는 얼마인가요?\n2. 데이터 소진 후 속도 제한이 적용되는 요금제는 어떤 것이 있나요?\n3. 장애인 혜택이 제공되는 요금제는 무엇인가요?\n4. 군인 전용 요금제는 어떤 것이 있나요?\n5. 월정액이 10만원 이하인 요금제는 어떤 것이 있나요?\n6. 데이터 리필 쿠폰을 선물할 수 있는 요금제는 무엇인가요?\n7. 음성 통화 제공량이 많은 요금제는 어떤 것이 있나요?\n8. SK텔레콤의 시니어 대상 요금제는 어떻게 되나요?\n9. 선택 약정 할인 포함 월정액이 얼마인가요?\n10. 가입 가능한 최대 나이는 몇 세인가요?\n11. 문자 제공량이 많은 요금제는 어떤 것이 있나요?\n12. 데이터 소진 후 제공되는 최대 금액은 얼마인가요?\n13. 다이렉트 플랜 가입이 가능한 요금제는 무엇인가요?\n14. SK텔레콤의 부가 서비스는 어떤 것이 있나요?\n15. 데이터 선물 받기가 가능한 요금제는 무엇인가요?\n16. 기본 제공 데이터 용량이 50GB 이상인 요금제는 어떤 것이 있나요?\n17. 후불 청구 방식의 요금제는 어떤 것이 있나요?\n18. 가입 가능한 최소 나이는 몇 세인가요?\n19. 영상 및 부가 통화 제공량이 많은 요금제는 무엇인가요?\n20. 동일 명의 가입이 가능한 요금제는 어떤 것이 있나요?\n21. T지원금 약정 동시 가입이 가능한 요금제는 무엇인가요?\n22. 데이터 소진 후 데이터 제공 속도가 어떤 요금제에서 가장 빠른가요?\n23. 부가세 제외 월정액이 8만원 이하인 요금제는 어떤 것이 있나요?\n24. SK텔레콤의 데이터 충전 서비스는 어떻게 이용하나요?\n25. 가입 동시 해지가 가능한 부가 서비스는 무엇인가요?\n26. 기본 제공 데이터 중 공유 가능한 용량이 많은 요금제는 어떤 것이 있나요?\n27. 장애인 부가 통화 추가 제공량이 있는 요금제는 무엇인가요?\n28. SK텔레콤의 요금제 그룹은 어떤 것들이 있나요?\n29. 데이터 리필 가능한 용량이 많은 요금제는 어떤 것이 있나요?\